In [ ]:
import os
import json
import time
import argparse
import warnings
warnings.filterwarnings("ignore")

import numpy as np

try:
    import cv2
except ImportError:
    raise SystemExit(
        "opencv-python is required for this script. Install with:\n"
        "  pip install opencv-python"
    )

try:
    import mediapipe as mp
except ImportError:
    raise SystemExit(
        "mediapipe is required for this script. Install with:\n"
        "  pip install mediapipe"
    )

import tensorflow as tf
from tensorflow.keras.models import load_model

SEQUENCE_LENGTH = 30
COUNTDOWN_SECONDS = 5.0  # matches test_webcam_trigger.py exactly

mp_holistic = mp.solutions.holistic
mp_drawing  = mp.solutions.drawing_utils

In [ ]:
# ============================================================
# Exact extraction logic from extraction.py / test_webcam_trigger.py
# ============================================================
SELECTED_FACE_IDS = [
    # Lips (For mouthing/shape)
    0, 13, 14, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84, 87, 88, 91, 95, 146,
    178, 181, 191, 267, 269, 270, 291, 308, 310, 311, 312, 314, 317, 318,
    321, 324, 375, 402, 405, 415,
    # Eyebrows
    46, 52, 53, 55, 65, 70, 105, 107, 276, 282, 283, 285, 295, 300, 334, 336,
    # Left Cheek Zone
    50, 118, 123, 137, 205, 206, 207, 212, 214, 216,
    # Right Cheek Zone
    280, 347, 352, 366, 425, 426, 427, 432, 434, 436
]


def extract_and_normalize_keypoints(results):
    """Verbatim from test_webcam_trigger.py / extraction.py."""
    cx, cy, cz = 0.0, 0.0, 0.0
    scale = 1.0

    if results.pose_landmarks:
        l_sh = results.pose_landmarks.landmark[11]
        r_sh = results.pose_landmarks.landmark[12]
        cx, cy, cz = (l_sh.x + r_sh.x) / 2, (l_sh.y + r_sh.y) / 2, (l_sh.z + r_sh.z) / 2
        shoulder_dist = np.linalg.norm([l_sh.x - r_sh.x, l_sh.y - r_sh.y, l_sh.z - r_sh.z])
        if shoulder_dist > 0:
            scale = shoulder_dist

    def norm(lm_list, is_face=False):
        if not lm_list:
            return np.zeros(len(SELECTED_FACE_IDS) * 3) if is_face else np.zeros(21 * 3)
        data = []
        for i, lm in enumerate(lm_list.landmark):
            if is_face and i not in SELECTED_FACE_IDS:
                continue
            data.extend([(lm.x - cx) / scale, (lm.y - cy) / scale, (lm.z - cz) / scale])
        return np.array(data)

    lh, rh = norm(results.left_hand_landmarks), norm(results.right_hand_landmarks)
    face = norm(results.face_landmarks, is_face=True)

    if results.pose_landmarks:
        pose = np.array([[(lm.x - cx) / scale, (lm.y - cy) / scale, (lm.z - cz) / scale]
                          for lm in results.pose_landmarks.landmark]).flatten()
    else:
        pose = np.zeros(33 * 3)

    return np.concatenate([pose, face, lh, rh])

In [ ]:
# ============================================================
# Compact 4-block derivation, mirroring the extraction repo
# ============================================================
# This step is NEW to Round 2 and it belongs inside the timed path. The model no
# longer eats the 447-dim vector above; it eats a 146-dim compact vector, so a
# real deployment has to run this between MediaPipe and inference. Timing it
# separately is what lets the paper state the derivation's true cost instead of
# quietly leaving it out.
#
# Inlined rather than imported, so this notebook runs standalone on whichever
# machine has a webcam, with no dependency on the extraction repo. Verified to
# match compact_features.derive_frame exactly on random and real frames.
_FACE_SORTED = sorted(SELECTED_FACE_IDS)
_SLOT = {mid: i for i, mid in enumerate(_FACE_SORTED)}
N_FACE = len(_FACE_SORTED)
POSE_END = 33 * 3
FACE_END = POSE_END + N_FACE * 3
DIM_RAW = FACE_END + 2 * 21 * 3
DIM_COMPACT = 146

BROW_LEFT = [_SLOT[i] for i in (276, 282, 283, 285, 295, 300, 334, 336)]
BROW_RIGHT = [_SLOT[i] for i in (46, 52, 53, 55, 65, 70, 105, 107)]
LIP_TOP, LIP_BOTTOM = _SLOT[13], _SLOT[14]
LIP_CORNERS = _SLOT[61], _SLOT[291]
LIMBS = ((11, 13), (13, 15), (12, 14), (14, 16))   # L upper, L fore, R upper, R fore


def _unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 1e-6 else np.zeros(3)


def derive_frame(frame):
    """One (447,) shoulder-normalized frame -> one (146,) compact frame."""
    pose = frame[0:POSE_END].reshape(33, 3)
    face = frame[POSE_END:FACE_END].reshape(N_FACE, 3)
    out = np.zeros(DIM_COMPACT)

    out[0:126] = frame[FACE_END:DIM_RAW]          # A: hands, unchanged
    if not pose.any():
        return out

    for k, (a, b) in enumerate(LIMBS):            # B: limb directions
        out[126 + 3 * k:129 + 3 * k] = _unit(pose[b] - pose[a])
    out[138] = np.arctan2(*(pose[11, :2] - pose[12, :2])[::-1])

    inter_ocular = np.linalg.norm(pose[2, :2] - pose[5, :2])
    if inter_ocular < 1e-6:
        return out

    dx, dy = pose[2, :2] - pose[5, :2]            # C: head orientation
    out[139] = np.arctan2(dy, dx)
    ear_dist = np.linalg.norm(pose[7, :2] - pose[8, :2])
    if ear_dist > 1e-6:
        out[140] = (pose[0, 0] - (pose[7, 0] + pose[8, 0]) / 2) / ear_dist
    out[141] = (pose[0, 1] - (pose[2, 1] + pose[5, 1]) / 2) / inter_ocular

    if not face.any():                            # D: facial NMS
        return out
    out[142] = abs(face[LIP_BOTTOM, 1] - face[LIP_TOP, 1]) / inter_ocular
    out[143] = np.linalg.norm(face[LIP_CORNERS[1], :2]
                              - face[LIP_CORNERS[0], :2]) / inter_ocular
    out[144] = (pose[2, 1] - face[BROW_LEFT, 1].mean()) / inter_ocular
    out[145] = (pose[5, 1] - face[BROW_RIGHT, 1].mean()) / inter_ocular
    return out


# Cheap guard against a silently wrong copy of the derivation. A zero frame must
# stay zero rather than divide by nothing, and limb vectors must be unit length.
_probe = np.zeros(DIM_RAW)
assert not derive_frame(_probe).any(), "derivation leaks a division on a blank frame"
_probe[0:POSE_END] = np.random.default_rng(0).normal(size=POSE_END)
_c = derive_frame(_probe)
assert _c.shape == (DIM_COMPACT,)
assert np.allclose(np.linalg.norm(_c[126:138].reshape(4, 3), axis=-1), 1.0), \
    "limb vectors are not unit length"

In [ ]:
# ============================================================
# Live capture loop (mirrors test_webcam_trigger.py's state machine)
# ============================================================
def capture_one_sequence(holistic):
    """Runs the SPACE -> countdown -> RECORDING flow exactly like
    test_webcam_trigger.py. Returns (sequence, processing_ms, wallclock_ms)
    or None if the user quit. Returns the COMPACT sequence, so what is timed here
    is the real deployment path: MediaPipe, shoulder normalisation, then the
    4-block derivation."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam (source 0).")

    state = "IDLE"
    countdown_start_time = 0
    sequence = []
    processing_time_ms = 0.0   # MediaPipe + shoulder normalization
    derive_time_ms = 0.0       # compact 4-block derivation, new in Round 2
    wallclock_start = None

    print("Press SPACE to start recording, 'q' to quit.")

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("Failed to read from webcam.")
                return None

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            if state == "RECORDING":
                t0 = time.perf_counter()
                results = holistic.process(image)
                keypoints = extract_and_normalize_keypoints(results)
                t1 = time.perf_counter()
                compact = derive_frame(keypoints)
                t2 = time.perf_counter()
                processing_time_ms += (t1 - t0) * 1000
                derive_time_ms += (t2 - t1) * 1000
                sequence.append(compact)
            else:
                results = holistic.process(image)  # still needed to draw / for UX

            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

            if state == "RECORDING":
                cv2.rectangle(image, (0, 0), (640, 40), (0, 0, 255), -1)
                cv2.putText(image, f'RECORDING: {len(sequence)}/{SEQUENCE_LENGTH} frames',
                            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
                if len(sequence) == SEQUENCE_LENGTH:
                    wallclock_ms = (time.perf_counter() - wallclock_start) * 1000
                    cap.release()
                    cv2.destroyAllWindows()
                    return (np.array(sequence, dtype=np.float32),
                            processing_time_ms, derive_time_ms, wallclock_ms)

            elif state == "IDLE":
                cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
                cv2.putText(image, "Press SPACE to start recording", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            elif state == "COUNTDOWN":
                time_left = COUNTDOWN_SECONDS - (time.time() - countdown_start_time)
                if time_left <= 0:
                    state = "RECORDING"
                    sequence = []
                    processing_time_ms = 0.0
                    derive_time_ms = 0.0
                    wallclock_start = time.perf_counter()
                else:
                    cv2.rectangle(image, (0, 0), (640, 80), (0, 165, 255), -1)
                    cv2.putText(image, f"GET READY: {int(time_left) + 1}", (10, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3, cv2.LINE_AA)

            cv2.imshow('SignLingo Latency Benchmark', image)
            key = cv2.waitKey(10) & 0xFF
            if key == ord('q'):
                cap.release()
                cv2.destroyAllWindows()
                return None
            elif key == ord(' ') and state == "IDLE":
                state = "COUNTDOWN"
                countdown_start_time = time.time()
    finally:
        cap.release()
        cv2.destroyAllWindows()

In [ ]:
# ============================================================
# Model timing (same captured sequence, every format)
# ============================================================
def time_keras_predict(h5_path, sequence, n_repeats=5):
    model = load_model(h5_path)
    inp = np.expand_dims(sequence, axis=0)
    latencies = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        model.predict(inp, verbose=0)
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
    return float(np.mean(latencies))


def time_tflite_predict(tflite_path, sequence, n_repeats=5):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    inp = sequence[np.newaxis, ...].astype(np.float32)

    latencies = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details[0]["index"], inp)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details[0]["index"])
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
    return float(np.mean(latencies))

In [ ]:
# ============================================================
# Main
# ============================================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_sequences", type=int, default=10,
                        help="Number of live sequences to capture and average over "
                             "(kept >1 so latency is reported as mean +/- std, not a "
                             "single sample)")
    parser.add_argument("--h5_model", default="models/signlingo_r2_best.h5",
                        help="Path to the raw Keras .h5 model")
    parser.add_argument("--results", default="final_results.json",
                        help="train_final output, read to learn the deployed arm width")
    args, _ = parser.parse_known_args()  # ignore Jupyter kernel's --f=... arg

    candidates_tflite = [
        ("Float32 (baseline)", "signlingo_r2_fp32.tflite"),
        ("Float16",            "signlingo_r2_fp16.tflite"),
        ("Dynamic Range",      "signlingo_r2_dynamic.tflite"),
        ("Full INT8",          "signlingo_r2_int8.tflite"),
    ]

    # The deployed model takes only its arm's prefix of the compact vector, so the
    # captured sequence is sliced to match. Read from train_final rather than typed
    # in here, so a different winning arm cannot silently produce a shape error.
    arm_cut, arm_name = DIM_COMPACT, "full compact vector"
    if os.path.exists(args.results):
        with open(args.results) as fh:
            _fr = json.load(fh)
        arm_cut, arm_name = int(_fr["dims"]), _fr["arm"]
        print(f"Deployed arm from {args.results}: {arm_name} ({arm_cut} dims)")
    else:
        print(f"  {args.results} not found; timing the full {DIM_COMPACT}-dim vector.")

    processing_times, derive_times, wallclock_times, sequences = [], [], [], []

    with mp_holistic.Holistic(min_detection_confidence=0.5,
                               min_tracking_confidence=0.5) as holistic:
        for i in range(args.n_sequences):
            print(f"\n--- Capturing sequence {i + 1}/{args.n_sequences} ---")
            result = capture_one_sequence(holistic)
            if result is None:
                print("Quit before completing capture.")
                return
            seq, proc_ms, der_ms, wall_ms = result
            sequences.append(seq[:, :arm_cut])
            processing_times.append(proc_ms + der_ms)
            derive_times.append(der_ms)
            wallclock_times.append(wall_ms)
            print(f"  MediaPipe + normalise: {proc_ms:.2f} ms  |  "
                  f"compact derivation: {der_ms:.2f} ms  |  "
                  f"capture wall-clock: {wall_ms:.2f} ms")

    mean_processing = float(np.mean(processing_times))
    std_processing  = float(np.std(processing_times))
    mean_derive     = float(np.mean(derive_times))
    std_derive      = float(np.std(derive_times))
    mean_wallclock  = float(np.mean(wallclock_times))

    print(f"\nProcessing time (compute only), N={len(processing_times)}: "
          f"{mean_processing:.2f} +/- {std_processing:.2f} ms")
    print(f"  of which the compact derivation is {mean_derive:.2f} +/- "
          f"{std_derive:.2f} ms ({mean_derive / mean_processing * 100:.1f}%)")
    print(f"Mean capture wall-clock time (incl. camera pacing): {mean_wallclock:.2f} ms\n")

    results = []

    def add_row(label, model_times):
        # Paired per-sequence totals (same N captured sequences), not a
        # combination of separately-averaged means -- gives a statistically
        # correct mean +/- std on the actual end-to-end number.
        totals = [p + m for p, m in zip(processing_times, model_times)]
        model_mean, model_std = float(np.mean(model_times)), float(np.std(model_times))
        total_mean, total_std = float(np.mean(totals)), float(np.std(totals))
        results.append({
            "Model": label,
            "N": len(model_times),
            "Arm": arm_name,
            "Arm Dims": arm_cut,
            "Processing Mean (ms)": round(mean_processing, 3),
            "Processing Std (ms)": round(std_processing, 3),
            "Compact Derivation Mean (ms)": round(mean_derive, 3),
            "Compact Derivation Std (ms)": round(std_derive, 3),
            "Model Inference Mean (ms)": round(model_mean, 3),
            "Model Inference Std (ms)": round(model_std, 3),
            "Total Processing Mean (ms)": round(total_mean, 3),
            "Total Processing Std (ms)": round(total_std, 3),
            "Total incl. Capture Wall-Clock (ms)": round(mean_wallclock + model_mean, 3),
        })
        print(f"  Model inference: {model_mean:.3f} +/- {model_std:.3f} ms  |  "
              f"Total: {total_mean:.3f} +/- {total_std:.3f} ms")

    if os.path.exists(args.h5_model):
        print(f"Timing raw Keras model: {args.h5_model}...")
        add_row("Raw Keras .h5 (eager)", [time_keras_predict(args.h5_model, seq) for seq in sequences])
    else:
        print(f"  Skipping raw Keras model: '{args.h5_model}' not found")

    for label, path in candidates_tflite:
        if not os.path.exists(path):
            print(f"  Skipping {label}: '{path}' not found")
            continue
        print(f"Timing model inference: {label}...")
        add_row(label, [time_tflite_predict(path, seq) for seq in sequences])

    print("\n== End-to-End Latency Results (mean +/- std) =================")
    header = (f"{'Model':<24}{'Processing (ms)':>20}{'Model (ms)':>18}{'Total (ms)':>20}")
    print(header)
    for r in results:
        proc = f"{r['Processing Mean (ms)']:.3f}+/-{r['Processing Std (ms)']:.3f}"
        mdl  = f"{r['Model Inference Mean (ms)']:.3f}+/-{r['Model Inference Std (ms)']:.3f}"
        tot  = f"{r['Total Processing Mean (ms)']:.3f}+/-{r['Total Processing Std (ms)']:.3f}"
        print(f"{r['Model']:<24}{proc:>20}{mdl:>18}{tot:>20}")

    print("\nProcessing covers MediaPipe, shoulder normalisation AND the compact")
    print("derivation, all three of which a real deployment runs per frame.")
    print(f"Model column is the forward pass on the {arm_cut}-dim {arm_name} vector.")

    # New filename on purpose. e2e_latency_results.json holds the old 447-dim
    # numbers, which are not comparable and are already flagged stale in the notes.
    with open("latency_r2_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print("\nSaved -> latency_r2_results.json")


if __name__ == "__main__":
    main()